# OCR model comparison

Reads every supported image in `IMAGE_DIR`, calls every entry in `MODEL_CONFIGS`, and creates an Excel workbook. The first column contains the embedded image. Each model gets four adjacent columns: extracted text, input tokens, output tokens, and estimated cost in USD.

**Privacy:** images are uploaded to remote providers. Use Ollama for local-only processing. A cloud model's cost remains blank until both prices are supplied.

Set API keys in the environment before running (for example `OPENAI_API_KEY`). Update pricing in `MODEL_CONFIGS`; provider APIs do not reliably return monetary cost.

In [ ]:
from pathlib import Path
import base64
import json
import os

import openpyxl
import requests
from openpyxl import Workbook, load_workbook
from openpyxl.drawing.image import Image as ExcelImage
from openpyxl.styles import Alignment, Font, PatternFill
from PIL import Image, ImageOps

print('Notebook imports loaded.')

## 1. Configure models

Add or remove entries freely. `name` becomes the Excel column prefix. Supported providers are `openai-compatible` and `ollama`. Prices are USD per one million tokens; enter current cloud prices or leave them as `None`. Local Ollama entries use `0.0`. `MAX_IMAGES` is useful for an inexpensive test run; set it to `None` for all images.

In [ ]:
IMAGE_DIR = Path('vlm_test')
OUTPUT_XLSX = Path('ocr_model_comparison.xlsx')
MAX_IMAGES = None

OCR_PROMPT = (
    'Extract all text from this image exactly as written. Preserve Tamil and other scripts, '
    'line breaks, punctuation, and numbers. Return only the transcription, with no commentary.'
)

MODEL_CONFIGS = [
    {
        'name': 'gpt-5-nano',
        'provider': 'openai-compatible',
        'model': 'gpt-5-nano',
        'base_url': 'https://api.openai.com/v1',
        'api_key_env': 'OPENAI_API_KEY',
        'input_usd_per_million': None,
        'output_usd_per_million': None,
        'timeout_seconds': 180,
    },
    {
        'name': 'gpt-5',
        'provider': 'openai-compatible',
        'model': 'gpt-5',
        'base_url': 'https://api.openai.com/v1',
        'api_key_env': 'OPENAI_API_KEY',
        'input_usd_per_million': None,
        'output_usd_per_million': None,
        'timeout_seconds': 180,
    },
    {
        'name': 'ollama-qwen2.5vl',
        'provider': 'ollama',
        'model': 'qwen2.5vl:7b',
        'base_url': 'http://localhost:11434',
        'input_usd_per_million': 0.0,
        'output_usd_per_million': 0.0,
        'timeout_seconds': 300,
    },
]

SUPPORTED_SUFFIXES = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tif', '.tiff'}
image_paths = sorted(
    p for p in IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES
) if IMAGE_DIR.exists() else []
if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]

print(f'Found {len(image_paths)} image(s) in {IMAGE_DIR.resolve()}')
print('\n'.join(str(p) for p in image_paths[:10]))

## 2. OCR clients and cost calculation

The OpenAI-compatible request uses the Responses API. It also works with compatible endpoints that expose the same API. Ollama uses `/api/chat`. Token counts are blank when an endpoint does not report them.

In [ ]:
def image_mime_and_data_uri(path: Path) -> tuple[str, str]:
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert('RGB')
        mime = 'image/jpeg'
        import io
        buffer = io.BytesIO()
        image.save(buffer, format='JPEG', quality=95)
    encoded = base64.b64encode(buffer.getvalue()).decode('ascii')
    return mime, f'data:{mime};base64,{encoded}'


def image_as_base64(path: Path) -> str:
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert('RGB')
        import io
        buffer = io.BytesIO()
        image.save(buffer, format='JPEG', quality=95)
    return base64.b64encode(buffer.getvalue()).decode('ascii')


def first_number(*values):
    for value in values:
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return int(value)
    return None


def extract_openai_compatible_result(payload: dict) -> tuple[str, int | None, int | None]:
    if 'error' in payload:
        raise RuntimeError(json.dumps(payload['error']))

    text = payload.get('output_text')
    if not isinstance(text, str):
        chunks = []
        for item in payload.get('output', []):
            for content in item.get('content', []):
                if content.get('type') in {'output_text', 'text'} and content.get('text'):
                    chunks.append(content['text'])
        if chunks:
            text = '\n'.join(chunks)

    if not isinstance(text, str):
        choice = (payload.get('choices') or [{}])[0]
        message = choice.get('message') or {}
        content = message.get('content', '')
        if isinstance(content, list):
            text = '\n'.join(part.get('text', '') for part in content if isinstance(part, dict))
        else:
            text = str(content)

    usage = payload.get('usage') or {}
    input_tokens = first_number(usage.get('input_tokens'), usage.get('prompt_tokens'))
    output_tokens = first_number(usage.get('output_tokens'), usage.get('completion_tokens'))
    return text or '', input_tokens, output_tokens


def call_openai_compatible(image_path: Path, config: dict) -> tuple[str, int | None, int | None]:
    api_key = os.getenv(config['api_key_env'], config.get('api_key', ''))
    if not api_key:
        raise RuntimeError(f"Set {config['api_key_env']} before using {config['name']}.")

    _, data_uri = image_mime_and_data_uri(image_path)
    endpoint = config['base_url'].rstrip('/') + '/responses'
    body = {
        'model': config['model'],
        'input': [{
            'role': 'user',
            'content': [
                {'type': 'input_text', 'text': OCR_PROMPT},
                {'type': 'input_image', 'image_url': data_uri},
            ],
        }],
    }
    response = requests.post(
        endpoint,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json=body,
        timeout=config.get('timeout_seconds', 180),
    )
    try:
        payload = response.json()
    except requests.exceptions.JSONDecodeError:
        response.raise_for_status()
        raise
    if not response.ok:
        raise RuntimeError(f'HTTP {response.status_code}: {json.dumps(payload)}')
    return extract_openai_compatible_result(payload)


def call_ollama(image_path: Path, config: dict) -> tuple[str, int | None, int | None]:
    body = {
        'model': config['model'],
        'stream': False,
        'messages': [{
            'role': 'user',
            'content': OCR_PROMPT,
            'images': [image_as_base64(image_path)],
        }],
        'options': config.get('options', {}),
    }
    response = requests.post(
        config['base_url'].rstrip('/') + '/api/chat',
        json=body,
        timeout=config.get('timeout_seconds', 300),
    )
    response.raise_for_status()
    payload = response.json()
    if 'error' in payload:
        raise RuntimeError(str(payload['error']))
    return (
        (payload.get('message') or {}).get('content', ''),
        first_number(payload.get('prompt_eval_count')),
        first_number(payload.get('eval_count')),
    )


def run_model(image_path: Path, config: dict) -> tuple[str, int | None, int | None]:
    provider = config['provider'].lower()
    if provider == 'openai-compatible':
        return call_openai_compatible(image_path, config)
    if provider == 'ollama':
        return call_ollama(image_path, config)
    raise ValueError(f"Unsupported provider: {config['provider']}")


def estimated_cost(config: dict, input_tokens: int | None, output_tokens: int | None) -> float | None:
    if input_tokens is None or output_tokens is None:
        return None
    input_price = config.get('input_usd_per_million')
    output_price = config.get('output_usd_per_million')
    if input_price is None or output_price is None:
        return None
    input_cost = input_tokens * input_price / 1_000_000
    output_cost = output_tokens * output_price / 1_000_000
    return input_cost + output_cost


## 3. Run all model/image combinations and build the workbook

Errors are written into the relevant model's text cell instead of stopping the remaining images and models.

In [ ]:
def prepare_excel_image(path: Path, target_width: int = 180) -> ExcelImage:
    import tempfile
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert('RGB')
        scale = min(1.0, target_width / image.width)
        image = image.resize((round(image.width * scale), round(image.height * scale)))
        cache_dir = Path(tempfile.gettempdir()) / 'ocr-model-comparison-images'
        cache_dir.mkdir(exist_ok=True)
        temporary_path = cache_dir / f'{path.stem}-{abs(hash(str(path.resolve())))}.png'
        image.save(temporary_path, format='PNG')
    return ExcelImage(temporary_path)


def write_comparison_workbook(image_paths: list[Path], configs: list[dict], output_path: Path):
    if not image_paths:
        raise FileNotFoundError(f'No supported images found in {IMAGE_DIR.resolve()}')
    if not configs:
        raise ValueError('MODEL_CONFIGS is empty.')
    names = [config['name'] for config in configs]
    if len(names) != len(set(names)):
        raise ValueError('Every MODEL_CONFIGS entry must have a unique name.')

    workbook = Workbook()
    sheet = workbook.active
    sheet.title = 'OCR comparison'
    header_fill = PatternFill('solid', fgColor='1F4E78')
    error_fill = PatternFill('solid', fgColor='FCE8E6')
    total_cost = 0.0

    headers = ['Image', 'File name']
    for config in configs:
        headers.extend([
            f"{config['name']} - text",
            f"{config['name']} - input tokens",
            f"{config['name']} - output tokens",
            f"{config['name']} - cost (USD)",
        ])
    sheet.append(headers)
    for cell in sheet[1]:
        cell.fill = header_fill
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    sheet.freeze_panes = 'C2'

    sheet.column_dimensions['A'].width = 26
    sheet.column_dimensions['B'].width = 28
    for model_index in range(len(configs)):
        base = 3 + model_index * 4
        sheet.column_dimensions[openpyxl.utils.get_column_letter(base)].width = 52
        for offset in (1, 2, 3):
            col_number = base + offset
            sheet.column_dimensions[openpyxl.utils.get_column_letter(col_number)].width = 18

    for image_index, image_path in enumerate(image_paths, start=2):
        print(f'[{image_index - 1}/{len(image_paths)}] {image_path.name}')
        excel_image = prepare_excel_image(image_path)
        sheet.add_image(excel_image, f'A{image_index}')
        sheet.cell(image_index, 2, image_path.name)
        sheet.row_dimensions[image_index].height = max(110, excel_image.height * 0.75)

        for model_index, config in enumerate(configs):
            base = 3 + model_index * 4
            print(f"  - {config['name']}", end=' ... ', flush=True)
            try:
                text, input_tokens, output_tokens = run_model(image_path, config)
                cost = estimated_cost(config, input_tokens, output_tokens)
                if cost is not None:
                    total_cost += cost
                sheet.cell(image_index, base, text)
                sheet.cell(image_index, base + 1, input_tokens)
                sheet.cell(image_index, base + 2, output_tokens)
                sheet.cell(image_index, base + 3, cost)
                sheet.cell(image_index, base + 3).number_format = '$0.000000'
                print('done')
            except Exception as exc:
                error_text = f'ERROR: {type(exc).__name__}: {exc}'
                sheet.cell(image_index, base, error_text).fill = error_fill
                print('failed')
                print(f'    {error_text}')

        for cell in sheet[image_index]:
            cell.alignment = Alignment(vertical='top', wrap_text=True)

    total_row = len(image_paths) + 3
    sheet.cell(total_row, 2, 'Estimated total cost').font = Font(bold=True)
    for model_index in range(len(configs)):
        cost_column = 6 + model_index * 4
        letter = openpyxl.utils.get_column_letter(cost_column)
        cell = sheet.cell(total_row, cost_column, f'=SUM({letter}2:{letter}{len(image_paths) + 1})')
        cell.number_format = '$0.000000'
        cell.font = Font(bold=True)

    sheet.auto_filter.ref = f'A1:{openpyxl.utils.get_column_letter(len(headers))}{len(image_paths) + 1}'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    workbook.save(output_path)
    return output_path.resolve(), total_cost

## 4. Run the comparison

Run this cell only after checking the model names, API keys, Ollama availability, and pricing.

In [ ]:
output_file, estimated_total = write_comparison_workbook(image_paths, MODEL_CONFIGS, OUTPUT_XLSX)
print(f'Wrote {output_file}')
print(f'Estimated priced cost: ${estimated_total:.6f}')

## 5. Optional: verify the workbook without calling models again

In [ ]:
check = load_workbook(OUTPUT_XLSX)
check_sheet = check['OCR comparison']
print('Workbook:', OUTPUT_XLSX.resolve())
print('Rows:', check_sheet.max_row, 'Columns:', check_sheet.max_column)
print('Embedded images:', len(check_sheet._images))
check.close()